# JAI fine-tuning on a free GPU (Google Colab or Kaggle)

Trains a LoRA adapter on **Qwen/Qwen2.5-7B-Instruct**, the model behind Ollama's `qwen2.5:7b-instruct`, using the human-reviewed JAI export.
At the end it produces a `q4_k_m` GGUF file you download and load on the Mac with `finetuning/install_candidate.sh`.

Before you start:
1. On the Mac, export the reviewed data and zip it: `cd finetuning/exports && zip -r reviewed-v3.zip reviewed-v3`.
2. Colab: **Runtime → Change runtime type → T4 GPU**. Kaggle: **Settings → Accelerator → GPU T4 x2**, Internet on, and add the zip as a dataset.
3. Run the cells in order. Training takes about 30–60 minutes; GGUF conversion another 15–25.

Only the fictional, reviewed training rows are uploaded. The held-out test file stays in the zip for provenance and is never trained on.

In [ ]:
import shutil
if not shutil.which('nvidia-smi'):
    raise SystemExit('No GPU attached. Colab: Runtime -> Change runtime type -> T4 GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2 (needs a verified phone). Then run again.')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 1. Upload and verify the reviewed export

In [ ]:
import hashlib, json, os, sys, zipfile, pathlib
if 'google.colab' in sys.modules:
    # Colab: an upload button appears under this cell. (Colab also has a /kaggle/input folder, so check Colab first.)
    from google.colab import files
    src = pathlib.Path(next(iter(files.upload())))
else:
    # Kaggle unzips attached datasets, so look for the zip or an already extracted provenance.json.
    found = list(pathlib.Path('/kaggle/input').rglob('reviewed-v*.zip')) + list(pathlib.Path('/kaggle/input').rglob('provenance.json'))
    if not found:
        raise SystemExit('No JAI dataset attached. Add Input -> Upload -> New Dataset with reviewed-v3.zip, then run this cell again.')
    src = found[0] if found[0].suffix == '.zip' else found[0].parent
if src.suffix == '.zip':
    zipfile.ZipFile(src).extractall('data'); data = next(pathlib.Path('data').rglob('provenance.json')).parent
else:
    data = src
provenance = json.loads((data / 'provenance.json').read_text())
for name, sha in provenance['files'].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == sha, f'{name} changed after export'
train_rows = [json.loads(l) for l in (data / 'train.jsonl').read_text().splitlines()]
print(len(train_rows), 'reviewed training examples from', data, '- hashes verified')


## 2. Load the base model in 4-bit and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch
MAX_LEN = 6144   # production prompts reach ~5k tokens (system prompt + website facts + operation list)
model, tokenizer = FastLanguageModel.from_pretrained('Qwen/Qwen2.5-7B-Instruct', max_seq_length=MAX_LEN, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)

## 3. Tokenise: loss only on JAI's final answer

The prompt (system, history, facts) is masked, so the model learns to *answer*, not to repeat the context.
Examples longer than `MAX_LEN` stop the run instead of being silently cut.

In [ ]:
from datasets import Dataset
def encode(row):
    prompt = tokenizer.apply_chat_template(row['prompt'], tokenize=True, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(row['prompt'] + row['completion'], tokenize=True)
    assert full[:len(prompt)] == prompt, 'chat template is not prefix-stable'
    if full[-1] != tokenizer.convert_tokens_to_ids('<|im_end|>'):
        full = full[:-1] if tokenizer.decode(full[-1:]).strip() == '' else full   # drop a trailing newline token only
    assert len(full) <= MAX_LEN, f'{len(full)} tokens > {MAX_LEN}'
    return {'input_ids': full, 'attention_mask': [1] * len(full), 'labels': [-100] * len(prompt) + full[len(prompt):]}
encoded = Dataset.from_list([encode(r) for r in train_rows])
lengths = [len(x) for x in encoded['input_ids']]
print('examples', len(lengths), 'max tokens', max(lengths), 'mean', sum(lengths) // len(lengths))

## 4. Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
args = TrainingArguments(output_dir='jai-lora-run', num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, warmup_steps=5, lr_scheduler_type='cosine', optim='adamw_8bit', weight_decay=0.01,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=1, save_strategy='no', report_to='none', seed=42)
trainer = Trainer(model=model, args=args, train_dataset=encoded,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
stats = trainer.train()
print(stats)

## 5. Quick look (a new question, not from the test set)

In [ ]:
FastLanguageModel.for_inference(model)
probe = train_rows[0]['prompt'][:1] + [{'role': 'user', 'content': json.dumps({'message': 'What should I check before generating a timetable?', 'facts': {}, 'pending': None, 'operations': []})}]
ids = tokenizer.apply_chat_template(probe, add_generation_prompt=True, return_tensors='pt').to('cuda')
print(tokenizer.decode(model.generate(input_ids=ids, max_new_tokens=200, do_sample=False)[0][ids.shape[1]:], skip_special_tokens=True))

## 6. Save the adapter and the GGUF for Ollama

In [ ]:
import platform, unsloth, transformers
model.save_pretrained('jai-lora'); tokenizer.save_pretrained('jai-lora')
model.save_pretrained_gguf('jai-gguf', tokenizer, quantization_method='q4_k_m')
# Newer Unsloth versions write the GGUF to /tmp/unsloth_saves instead of the working folder.
ggufs = [p for d in ('.', '/tmp/unsloth_saves') if pathlib.Path(d).exists() for p in pathlib.Path(d).rglob('*.gguf')]
gguf = max((p for p in ggufs if 'Q4_K_M' in p.name.upper()), key=lambda p: p.stat().st_size, default=None) or max(ggufs, key=lambda p: p.stat().st_size)
record = {'base_model': 'Qwen/Qwen2.5-7B-Instruct', 'dataset': provenance, 'examples': len(train_rows), 'max_tokens': max(lengths),
          'epochs': 2, 'learning_rate': 2e-4, 'lora_r': 16, 'final_loss': stats.training_loss, 'gpu': torch.cuda.get_device_name(0),
          'versions': {'unsloth': unsloth.__version__, 'transformers': transformers.__version__, 'torch': torch.__version__},
          'gguf': gguf.name, 'gguf_sha256': hashlib.sha256(gguf.read_bytes()).hexdigest()}
pathlib.Path('jai-lora/training-provenance.json').write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
!zip -qr jai-lora.zip jai-lora

## 7. Download

Colab: the cell below copies the GGUF (about 4.7 GB) and the LoRA zip to **Google Drive → My Drive → jai-finetune**; download them from there.
Kaggle: the files are under **Output** once you *Save Version → Save & Run All*.


In [ ]:
import shutil, sys
if 'google.colab' in sys.modules:
    # A 4.7 GB browser download from Colab often stalls, so copy both files to Google Drive (allow access when asked).
    from google.colab import drive
    drive.mount('/content/drive')
    out = pathlib.Path('/content/drive/MyDrive/jai-finetune'); out.mkdir(parents=True, exist_ok=True)
    for f in ['jai-lora.zip', str(gguf)]:
        shutil.copy(f, out / pathlib.Path(f).name)
        print('copied', f, '->', out / pathlib.Path(f).name)
    print('Open drive.google.com -> My Drive -> jai-finetune and download both files to the Mac Downloads folder.')
else:
    print('Kaggle: the files are under Output after Save Version -> Save & Run All.')


## 8. On the Mac

```bash
bash finetuning/install_candidate.sh ~/Downloads/<file>.gguf jai-candidate
backend/.venv/bin/python finetuning/toolkit.py evaluate --model jai-candidate --task conversation --limit 40 --timeout 200 --output finetuning/runs/tuned-conversation.jsonl
```
Compare with the same command for `qwen2.5:7b-instruct`, then build the blind rating sheet. See `finetuning/README.md`.